# 29_diffusion_basics.ipynb

**14주차 · 1교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`14week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 8셀. 위에서부터 순서대로 실행합니다.

## 2. 실습 1 — 순방향 노이즈 추가 직접 구현 ★

**셀 1** — 데이터와 스케줄

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

ds = datasets.FashionMNIST("data", train=True,
        transform=transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize((0.5,), (0.5,))]))  # [-1,1]
x0, y0 = ds[0]
x0 = x0.unsqueeze(0)                                    # (1, 1, 28, 28)
print("원본 :", x0.shape, "| 범위", x0.min().item(), "~", x0.max().item())

T = 1000
betas       = torch.linspace(1e-4, 0.02, T)             # ★ 선형 스케줄
alphas      = 1.0 - betas
alphas_bar  = torch.cumprod(alphas, dim=0)              # ᾱt
print("β  :", betas[0].item(), "→", betas[-1].item())
print("ᾱ  :", alphas_bar[0].item(), "→", alphas_bar[-1].item(), " ← 0 에 수렴 ★")

**셀 2** — 핵심 함수 ★★ 오늘의 하이라이트

In [ ]:
def q_sample(x0, t, alphas_bar):
    """x0 에 t 스텝만큼의 노이즈를 한 번에 더한다 → (x_t, ε)"""
    eps  = torch.randn_like(x0)                          # 정답이 될 노이즈
    ab   = alphas_bar[t].view(-1, 1, 1, 1)               # (B,1,1,1) 브로드캐스팅
    x_t  = ab.sqrt() * x0 + (1 - ab).sqrt() * eps        # ★ 공식 그대로
    return x_t, eps

xt, eps = q_sample(x0, torch.tensor([500]), alphas_bar)
print("x_500 :", xt.shape, "| 범위", round(xt.min().item(), 2), "~", round(xt.max().item(), 2))

**셀 3** — t 를 키우며 눈으로 본다 ★

In [ ]:
ts = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(2 * len(ts), 2.6))
for k, t in enumerate(ts):
    xt, _ = q_sample(x0, torch.tensor([t]), alphas_bar)
    axes[k].imshow(xt[0, 0], cmap="gray"); axes[k].axis("off")
    axes[k].set_title(f"t={t}\nᾱ={alphas_bar[t]:.3f}", fontsize=9)
plt.suptitle("순방향 과정 — 깨끗한 이미지가 잡음이 되어 간다")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★★**: ① 초반에는 **거의 변화가 없어 보입니다.** ② 중반(t≈300~600)에서 **급격히 무너집니다.** ③ 마지막에는 **원본을 전혀 알아볼 수 없습니다** — 순수한 가우시안 잡음입니다. **이 비선형성**이 실습 2에서 볼 스케줄 이야기의 출발점입니다.

**셀 4** — "노이즈를 알면 원본을 복원할 수 있다" ★

In [ ]:
t = torch.tensor([300])
xt, eps = q_sample(x0, t, alphas_bar)
ab = alphas_bar[t].view(-1, 1, 1, 1)

x0_hat = (xt - (1 - ab).sqrt() * eps) / ab.sqrt()        # ★ 공식을 x0 에 대해 푼 것
print("복원 오차 :", (x0_hat - x0).abs().max().item(), " ← 거의 0 ★")

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for a, im, ti in zip(axes, [x0, xt, x0_hat], ["원본 x₀", "노이즈 x₃₀₀", "ε 로 복원한 x₀"]):
    a.imshow(im[0, 0], cmap="gray"); a.axis("off"); a.set_title(ti)
plt.tight_layout(); plt.show()

> **핵심 ★★**: **이것이 확산 모델의 전부입니다.** `ε` 만 알면 `x₀` 를 되돌릴 수 있습니다. 그런데 실제로는 `ε` 를 모릅니다. **그래서 신경망이 `ε` 를 예측하도록 학습시키는 것**입니다. *"모델은 그림을 그리는 법이 아니라 노이즈를 알아보는 법을 배웁니다."*

## 3-1. 학습 알고리즘

**셀 5** — 학습 루프의 뼈대 (읽기만 ★)

In [ ]:
# 실제 학습은 하지 않는다. 구조만 읽는다.
"""
for x0, _ in dataloader:                                    # ★ 레이블 안 씀
    t   = torch.randint(0, T, (x0.size(0),))                # 스텝을 무작위로
    xt, eps = q_sample(x0, t, alphas_bar)                   # 노이즈 섞기 (실습 1)
    eps_hat = unet(xt, t)                                   # ★ 노이즈를 예측
    loss = F.mse_loss(eps_hat, eps)                         # ★ 그냥 MSE
    loss.backward(); opt.step(); opt.zero_grad()
"""
print("학습 = (노이즈 섞인 이미지, 스텝) → 노이즈 예측.  회귀 문제 하나뿐이다 ★")

## 4. 실습 2 — 노이즈 스케줄 시각화

**셀 6** — β 와 ᾱ 가 어떻게 변하나

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(betas);      axes[0].set_title("β_t — 매 스텝 넣는 노이즈 양"); axes[0].set_xlabel("t")
axes[1].plot(alphas_bar); axes[1].set_title("ᾱ_t — 원본이 남은 비율 ★");    axes[1].set_xlabel("t")
axes[2].plot(alphas_bar.sqrt(),        label="√ᾱ  (원본 비중)")
axes[2].plot((1 - alphas_bar).sqrt(),  label="√(1-ᾱ) (노이즈 비중)")
axes[2].legend(); axes[2].set_title("원본 vs 노이즈 비중"); axes[2].set_xlabel("t")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: `β` 는 **선형으로 조금씩** 커지는데, `ᾱ`(원본이 남은 비율)는 **곱셈이 누적되어 급격히 0 으로** 떨어집니다. 그래서 실습 1에서 **중반에 갑자기 무너져 보였던** 것입니다.

**셀 7** — 균등하게 더하면 왜 안 되나 ★

In [ ]:
lin  = torch.linspace(1e-4, 0.02, T)                        # 선형(기본)
flat = torch.full((T,), 0.02)                               # ★ 처음부터 크게 = 균등
ab_lin  = torch.cumprod(1 - lin,  0)
ab_flat = torch.cumprod(1 - flat, 0)

plt.figure(figsize=(7, 4))
plt.plot(ab_lin,  label="선형 스케줄 (기본)")
plt.plot(ab_flat, label="큰 β 로 균등 — 너무 빨리 망가진다 ★")
plt.axhline(0.5, color="gray", ls=":", label="원본이 절반 남는 지점")
plt.xlabel("t"); plt.ylabel("ᾱ_t"); plt.legend(); plt.title("노이즈 스케줄 비교")
plt.tight_layout(); plt.show()

for name, ab in [("선형", ab_lin), ("균등(큰 β)", ab_flat)]:
    half = (ab < 0.5).nonzero()[0].item()
    print(f"{name:12s} : t={half} 에서 원본이 절반으로 떨어진다")

> **핵심 ★★ (기말 출제 지점)**: **스케줄이 필요한 이유** — 노이즈를 **처음부터 크게** 넣으면 이미지가 **초반에 다 망가져** 대부분의 스텝이 낭비됩니다. **끝까지 균등하게 작게** 넣으면 T 스텝을 다 써도 완전한 잡음이 되지 않습니다. 그래서 **처음엔 조금, 뒤로 갈수록 많이** 넣어 **정보가 고르게 사라지도록** 설계합니다.

**셀 8** — 스텝 수를 줄인다는 것의 의미 (2교시 복선)

In [ ]:
for n_steps in [1000, 50, 20, 4]:
    idx = torch.linspace(0, T - 1, n_steps).long()
    print(f"{n_steps:5d} 스텝 → 한 번에 건너뛰는 t 간격 평균 {float((T-1)/n_steps):.1f}")
print("\n간격이 클수록 한 스텝에서 지워야 할 노이즈가 많다 → 품질 ↓, 속도 ↑ ★")